# Upscale a video with SeedVR2

From **The Token Desk**. The slowest of the upscalers we tested and the one that adds the most fine texture: skin, lashes, the weave of a suit.

**In our test** (a 16-second 480x854 clip to 1080x1920): 6.4 minutes on a Colab G4, about 4 compute units per minute of finished video; 35 minutes on an L4, about 3.7 units per minute. It used up to 22 GB of GPU memory, so a 24 GB card like an RTX 4090 runs it.

**How to run it:** Runtime → Change runtime type → G4 (fastest), A100 or L4 → Save. Then Runtime → Run all. Colab's free T4 is far too slow for it.

SeedVR2 is by ByteDance Seed; this runs numz's command-line version, Apache-2.0: [github.com/numz/ComfyUI-SeedVR2_VideoUpscaler](https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler).

In [ ]:
#@title 1. Check the GPU
import subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU yet. Runtime > Change runtime type > pick a GPU, Save, then run this cell again.")
gpu_name, gpu_mem, gpu_cap = [x.strip() for x in gpu.split(",")]
gpu_gb, gpu_cap = int(gpu_mem.split()[0]) / 1024, float(gpu_cap)
print(f"GPU: {gpu_name}, {gpu_gb:.0f} GB")
if gpu_gb < 20:
    raise SystemExit('SeedVR2 needs about 22 GB of GPU memory. Runtime > Change runtime type > L4 (slow), A100 or G4.')
if gpu_cap < 8:
    raise SystemExit("SeedVR2 is far too slow on this GPU: on Colab's T4 it had not finished 6 seconds of video after 75 minutes. Pick an L4, A100 or G4.")
NEED_GB = 22
if gpu_gb < NEED_GB:
    print(f"This model used up to {NEED_GB} GB in our test, so it may run out of memory here. If it does, pick a bigger GPU.")

In [ ]:
#@title 2. Your video
#@markdown Leave it as **sample** to try our 6-second sample clip. Clear it to upload a video from your computer, or paste a link to an .mp4.
VIDEO = "sample"  #@param {type:"string"}
import os, shutil, subprocess, urllib.request
os.makedirs("/content/in", exist_ok=True)
SAMPLE = "https://raw.githubusercontent.com/thetokendesk/notebooks/main/upscale-video/sample/sample-360p.mp4"
if VIDEO.strip() == "":
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    src = "/content/in/" + name
    shutil.move(name, src)
elif VIDEO.strip() == "sample" or VIDEO.strip().startswith("http"):
    url = SAMPLE if VIDEO.strip() == "sample" else VIDEO.strip()
    src = "/content/in/" + os.path.basename(url.split("?")[0])
    urllib.request.urlretrieve(url, src)
else:
    src = VIDEO.strip()
w, h = [int(x) for x in subprocess.run(
    f'ffprobe -v error -select_streams v:0 -show_entries stream=width,height -of csv=p=0 "{src}"',
    shell=True, capture_output=True, text=True).stdout.strip().split(",")[:2]]
seconds = float(subprocess.run(f'ffprobe -v error -show_entries format=duration -of csv=p=0 "{src}"',
                               shell=True, capture_output=True, text=True).stdout.strip())
print(f"Video: {os.path.basename(src)}, {w}x{h}, {seconds:.1f} s")

In [ ]:
#@title 3. Install SeedVR2 (about a minute)
import time
t0 = time.time()
REPO = "/content/SeedVR2"
COMMIT = "4490bd1"  # the version we tested
!git clone -q https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler {REPO} && cd {REPO} && git checkout -q {COMMIT}
!grep -v -i -E '^(torch|torchvision|torchaudio)([<>=~ ]|$)' {REPO}/requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
!hf download numz/SeedVR2_comfyUI seedvr2_ema_3b_fp16.safetensors ema_vae_fp16.safetensors --local-dir /content/weights
print(f"Installed in {(time.time() - t0) / 60:.1f} min")

In [ ]:
#@title 4. Upscale
RESOLUTION = 1080  #@param {type:"integer"}
#@markdown The short side of the result, in pixels: 1080 makes a vertical clip 1080x1920.

import threading, time
def watch_gpu_memory():
    """Peak GPU memory while the upscale runs, read every 2 s."""
    peak, stop = [0], threading.Event()
    def loop():
        while not stop.is_set():
            try:
                used = subprocess.run("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                                      shell=True, capture_output=True, text=True).stdout.split()[0]
                peak[0] = max(peak[0], int(used))
            except Exception:
                pass
            time.sleep(2)
    threading.Thread(target=loop, daemon=True).start()
    return peak, stop

os.makedirs("/content/out", exist_ok=True)
out = f"/content/out/{os.path.splitext(os.path.basename(src))[0]}-seedvr2.mp4"
peak, stop = watch_gpu_memory()
t0 = time.time()
!cd {REPO} && python inference_cli.py "{src}" --output /content/out/raw.mp4 --model_dir /content/weights --dit_model seedvr2_ema_3b_fp16.safetensors --resolution {RESOLUTION} --batch_size 33 --uniform_batch_size --temporal_overlap 4 --color_correction lab --vae_encode_tiled --vae_decode_tiled --video_backend ffmpeg
stop.set()
!ffmpeg -v error -y -i /content/out/raw.mp4 -i "{src}" -map 0:v -map 1:a? -c:v copy -c:a copy -shortest "{out}"
assert os.path.exists(out), "The upscale did not finish: read the lines above for the error."
print(f"Done in {(time.time() - t0) / 60:.1f} min, up to {peak[0] / 1024:.1f} GB of GPU memory.")

In [ ]:
#@title 5. Compare and download
CLOSE_UP = "upper"  #@param ["upper", "center", "lower"]
#@markdown Where the close-up is taken: **upper** suits a face in a vertical video.
from IPython.display import Image, display
at = min(2.0, seconds / 2)
OW, OH = [int(x) for x in subprocess.run(
    f'ffprobe -v error -select_streams v:0 -show_entries stream=width,height -of csv=p=0 "{out}"',
    shell=True, capture_output=True, text=True).stdout.strip().split(",")[:2]]
# The same moment, the original resized the ordinary way to the new size, and the upscale.
!ffmpeg -v error -y -ss {at} -i "{src}" -frames:v 1 -vf "scale={OW}:{OH}:flags=bicubic" /content/out/before.png
!ffmpeg -v error -y -ss {at} -i "{out}" -frames:v 1 /content/out/after.png
!ffmpeg -v error -y -i /content/out/before.png -i /content/out/after.png -filter_complex "[0]scale=-2:900[a];[1]scale=-2:900[b];[a][b]hstack" /content/out/compare.png
side = min(OW, OH) // 3
x = (OW - side) // 2
y = max(0, min(OH - side, int(OH * {"upper": 0.3, "center": 0.5, "lower": 0.7}[CLOSE_UP]) - side // 2))
!ffmpeg -v error -y -i /content/out/before.png -i /content/out/after.png -filter_complex "[0]crop={side}:{side}:{x}:{y},scale=600:600:flags=neighbor[a];[1]crop={side}:{side}:{x}:{y},scale=600:600:flags=neighbor[b];[a][b]hstack" /content/out/close-up.png
print("Left: your video, resized the ordinary way. Right: upscaled.")
display(Image("/content/out/compare.png", width=900))
print("Close-up:")
display(Image("/content/out/close-up.png", width=900))
try:
    from google.colab import files
    files.download(out)
except Exception:
    print("Download it from the Files panel on the left:", out)

### When you're done
Download your video first: everything here is deleted when the session ends. Then **Runtime → Disconnect and delete runtime**, so the GPU stops using your compute units.